# Pandas 01 - Pandas Foundations

> **MLCourse · Data Science Foundations · 02_pandas**

This is the first stop on our pandas journey - the notebook every other pandas
notebook builds on. If NumPy gave us fast *arrays*, pandas gives us fast
*labeled tables*: think "Excel or a SQL table, but programmable, reproducible,
and vastly more powerful."

## What you'll learn

- Why pandas exists and what problems it solves over raw NumPy
- Creating `Series` and `DataFrame`s by hand
- Reading data with `read_csv` (and seaborn's built-in datasets)
- The inspection toolkit: `head`, `info`, `describe`, `value_counts`, ...
- Selecting data three ways: `[]`, `.loc` (labels), `.iloc` (positions)
- Boolean filtering: `&` / `|` / `~`, `isin`, `between`, `query`
- Sorting (`sort_values`, `sort_index`) and ranking (`rank`)
- Adding derived columns, dropping/renaming, index surgery

In [1]:
# Setup cell - everything later assumes these imports exist.
# `%matplotlib inline` is Jupyter magic; we activate it safely so this file
# ALSO runs as a plain Python script (where get_ipython() does not exist).
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass  # not inside Jupyter - nothing to do

import numpy as np                      # numeric arrays, random numbers
import pandas as pd                     # the star of this course
import seaborn as sns                   # free practice datasets
import matplotlib.pyplot as plt         # plotting (used sparingly here)

np.random.seed(42)                      # reproducibility: same "random" every run
pd.set_option("display.max_columns", 120)  # don't hide columns when printing


def _load(name, demo_builder):
    """Try to download a seaborn dataset; if offline, fall back to a tiny
    hand-built stand-in so every cell below still runs end-to-end."""
    try:
        return sns.load_dataset(name)
    except Exception as e:
        print(f"[setup] '{name}' unavailable ({type(e).__name__}) -> using demo rows.")
        return demo_builder()


def _demo_titanic():
    """A miniature titanic-shaped table used ONLY when we cannot download."""
    return pd.DataFrame(
        {
            "survived": [0, 1, 1, 1, 0, 0, 1, 0, 1, 0],
            "pclass":   [3, 1, 1, 1, 3, 3, 1, 3, 2, 3],       # ticket class 1-3
            "sex":      ["male", "female", "female", "female",
                         "male", "male", "male", "male", "female", "female"],
            "age":      [22.0, 38.0, 35.0, 27.0, 35.0, np.nan, 54.0, 2.0, 58.0, np.nan],
            "fare":     [7.25, 71.28, 53.10, 57.00, 8.05, 8.46, 51.86, 21.07, 26.55, 7.75],
            "embarked": ["S", "C", "S", "S", "S", "Q", "S", "S", "C", "S"],
        }
    )


# Our running example for the whole notebook:
try:
    titanic = sns.load_dataset("titanic")           # 891 real passenger rows
except Exception as e:
    print("Could not download dataset (offline?):", e)
    titanic = _demo_titanic()                       # tiny fallback (10 rows)
titanic  # last expression in a cell is displayed automatically in Jupyter

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


## 1. Why pandas?

Raw NumPy arrays are great for numbers, but real datasets have:

- **labels** (column names, meaningful row indexes) instead of position 0..n
- **heterogeneous columns** (numbers, text, dates, categories side by side)
- **messiness** (missing values, duplicates, mixed types in one column)

pandas bundles all three concerns into one object model. A `Series` is a
single labeled column; a `DataFrame` is a table of Series sharing one index.
Everything else in this notebook is about building, inspecting, and slicing
those two objects.

### 1.1 Creating a Series

In [2]:
# WHAT: a Series = 1-D data + an index (the labels on the left).
# WHY:  the index lets us reference data by MEANING ("2024-Q1"), not by position.

# From a plain list -> pandas assigns a default RangeIndex 0,1,2,...
temps = pd.Series([21.5, 22.1, 19.8, 23.4])
print(temps, "\n")

# From a dict -> keys become the (custom) index labels.
pop = pd.Series({"Tokyo": 37.4, "Delhi": 32.9, "Shanghai": 29.2})
print(pop, "\n")

# Scalar + explicit index -> the scalar is broadcast to every label.
profit = pd.Series(1000, index=["Q1", "Q2", "Q3", "Q4"])
print(profit)

0    21.5
1    22.1
2    19.8
3    23.4
dtype: float64 

Tokyo       37.4
Delhi       32.9
Shanghai    29.2
dtype: float64 

Q1    1000
Q2    1000
Q3    1000
Q4    1000
dtype: int64


### 1.2 Peeking inside a Series

In [3]:
# .values -> the raw NumPy array underneath (positions only, no labels).
print("values :", temps.values)

# .index -> the label machinery (a RangeIndex here because we passed a list).
print("index  :", temps.index)

# .dtype -> ONE dtype per Series (that's why mixed-type data lands in 'object').
print("dtype  :", temps.dtype)

# Vectorized ops apply to EVERY element without a loop -- fast and readable.
print("\ntemps + 2.0:\n", temps + 2.0)
print("\nnumpy ufuncs work too:\n", np.round(temps))

values : [21.5 22.1 19.8 23.4]
index  : RangeIndex(start=0, stop=4, step=1)
dtype  : float64

temps + 2.0:
 0    23.5
1    24.1
2    21.8
3    25.4
dtype: float64

numpy ufuncs work too:
 0    22.0
1    22.0
2    20.0
3    23.0
dtype: float64


### 1.3 Alignment: the superpower (and the classic gotcha)

In [4]:
# When two Series interact, pandas aligns them BY INDEX LABEL first,
# then applies the operation label-by-label.
s1 = pd.Series([1, 2, 3], index=["a", "b", "c"])
s2 = pd.Series([10, 20, 30], index=["b", "c", "d"])   # note: 'a' and 'd' differ

s_sum = s1 + s2
print(s_sum)

# ⚠️ Common pitfall: labels present in only ONE operand become NaN --
# arithmetic with NaN propagates NaN. Not a bug: alignment working as designed!

# 💡 Pro tip: most operators have a method that accepts fill_value, letting us
# treat "missing on one side" as 0 during the operation.
print()
print(s1.add(s2, fill_value=0))   # 'a' gets 1+0, 'd' gets 0+30

a     NaN
b    12.0
c    23.0
d     NaN
dtype: float64

a     1.0
b    12.0
c    23.0
d    30.0
dtype: float64


## 2. Building DataFrames

WHAT: a DataFrame is a dict-like container of columns sharing one index.
WHY:  it mirrors how tabular data actually looks (rows × named columns).

In [5]:
# From a dict of lists: keys -> column names, lists -> column contents.
students = pd.DataFrame(
    {
        "name": ["Ann", "Ben", "Cal", "Dee"],
        "age":  [23, 31, 19, 27],
        "grade": [88.5, 72.0, 95.5, 67.0],
    }
)
print(students, "\n")

# From a dict of Series: watch ALIGNMENT happen again -- 'Berlin' exists only
# on one side, so pandas fills the union of labels with NaN where needed.
pop_2010 = pd.Series({"Tokyo": 36.8, "Delhi": 22.0, "Shanghai": 20.3})
pop_2025 = pd.Series({"Tokyo": 37.4, "Delhi": 32.9, "Lagos": 16.6})
growth = pd.DataFrame({"y2010": pop_2010, "y2025": pop_2025})
print(growth)   # Shanghai has 2010-only, Lagos has 2025-only -> NaN elsewhere

  name  age  grade
0  Ann   23   88.5
1  Ben   31   72.0
2  Cal   19   95.5
3  Dee   27   67.0 

          y2010  y2025
Delhi      22.0   32.9
Lagos       NaN   16.6
Shanghai   20.3    NaN
Tokyo      36.8   37.4


## 3. Reading data: `read_csv` and friends

WHAT: `read_csv` turns delimited text into a DataFrame.
WHY:  90% of real-world files arrive as CSV, and the function's many knobs
let us fix messiness AT LOAD TIME instead of patching afterwards.

In [6]:
from io import StringIO   # lets us pretend a string is a file -- perfect for demos

csv_text = """id,name,city,score,joined
1,Alice,London,88,2021-03-14
2,Bob,Paris,N/A,2020-11-02
3,Carla,Berlin,79.5,?
4,Dan,Dublin,91.25,2022-01-31
5,Eve,,66.0,2021-07-04
"""

survey = pd.read_csv(
    StringIO(csv_text),         # wrap the string so read_csv can "read" it
    sep=",",                    # field separator (use ";" or "\t" when needed)
    header=0,                   # row 0 holds the column names (the default)
    index_col="id",             # use the 'id' column as row labels
    dtype={"score": "float64"}, # force a dtype instead of guessing
    na_values=["N/A", "?"],     # extra strings to recognize as missing
    parse_dates=["joined"],     # convert date-looking columns to datetime64
)
print(survey, "\n")
print(survey.dtypes)            # score is float, joined is datetime64 -- clean!

     name    city  score     joined
id                                 
1   Alice  London  88.00 2021-03-14
2     Bob   Paris    NaN 2020-11-02
3   Carla  Berlin  79.50        NaT
4     Dan  Dublin  91.25 2022-01-31
5     Eve     NaN  66.00 2021-07-04 

name                 str
city                 str
score            float64
joined    datetime64[us]
dtype: object


Above we loaded **titanic** (891 rows in the full version) - it is our running
example: one row per passenger, mixing numbers, text, categories, and missing
values. Realistic enough to hurt, small enough to inspect.

### 3.1 First look at the running example

In [7]:
print(titanic.shape)          # (rows, columns)
print(titanic.columns.tolist())
titanic.head()                # first 5 rows; head(10) would give ten

(891, 15)
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 4. The inspection toolkit

WHAT: a handful of one-liners that answer "what did I just load?"
WHY:  ALWAYS inspect before analyzing - wrong dtypes, surprise missing
values, or phantom duplicates ruin downstream results silently.

In [8]:
# Quick peeks: head (top), tail (bottom), sample (random rows).
# random_state makes 'random' reproducible - same rows every run.
print(titanic.tail(3), "\n")
print(titanic.sample(5, random_state=42))

     survived  pclass     sex   age  sibsp  parch   fare embarked  class  \
888         0       3  female   NaN      1      2  23.45        S  Third   
889         1       1    male  26.0      0      0  30.00        C  First   
890         0       3    male  32.0      0      0   7.75        Q  Third   

       who  adult_male deck  embark_town alive  alone  
888  woman       False  NaN  Southampton    no  False  
889    man        True    C    Cherbourg   yes   True  
890    man        True  NaN   Queenstown    no   True   

     survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
709         1       3    male   NaN      1      1  15.2458        C   Third   
439         0       2    male  31.0      0      0  10.5000        S  Second   
840         0       3    male  20.0      0      0   7.9250        S   Third   
720         1       2  female   6.0      0      1  33.0000        S  Second   
39          1       3  female  14.0      1      0  11.2417        C   Thir

In [9]:
# Shape / size / column names / dtypes -- the structural X-ray.
print("shape :", titanic.shape)                 # tuple (n_rows, n_cols)
print("size  :", titanic.size)                  # n_rows * n_cols cells total
print("index :", titanic.index)                 # row-label machinery
print("\ndtypes:\n", titanic.dtypes)            # one dtype per column

shape : (891, 15)
size  : 13365
index : RangeIndex(start=0, stop=891, step=1)

dtypes:
 survived          int64
pclass            int64
sex                 str
age             float64
sibsp             int64
parch             int64
fare            float64
embarked            str
class          category
who                 str
adult_male         bool
deck           category
embark_town         str
alive               str
alone              bool
dtype: object


In [10]:
# info() = the single best first move on ANY new dataset:
# row count, dtypes, and NON-null counts per column (so gaps pop out).
titanic.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [11]:
# describe(): numeric summaries by default; include="all" adds text/category
# columns (their 'summary' is count/unique/top/freq instead of mean/std).
(
    titanic[["age", "fare", "sex", "embarked"]]  # subset keeps output readable
    .describe(include="all")
)

,age,fare,sex,embarked
count,714.000000,891.000000,891,889
unique,NaN,NaN,2,3
top,NaN,NaN,male,S
freq,NaN,NaN,577,644
mean,29.699118,32.204208,NaN,NaN
std,14.526497,49.693429,NaN,NaN
min,0.420000,0.000000,NaN,NaN
25%,20.125000,7.910400,NaN,NaN
50%,28.000000,14.454200,NaN,NaN
75%,38.000000,31.000000,NaN,NaN


In [12]:
# value_counts: frequency table of a Series. dropna=False shows the NaN bucket
# too -- by default missing values are hidden, which hides problems!
print(titanic["embarked"].value_counts(dropna=False), "\n")

# nunique: how many DISTINCT values (great sanity check for ID-like columns).
print("distinct ports:", titanic["embarked"].nunique())

# memory_usage(deep=True): honest byte count including strings; useful early.
mb = titanic.memory_usage(deep=True).sum() / 1024**2
print(f"titanic uses about {mb:.2f} MB")

embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64 

distinct ports: 3
titanic uses about 0.31 MB


> 💡 **Pro tip:** make `df.info()` + `df.describe(include='all')` +
> `value_counts(dropna=False)` your reflexive opening trio on every dataset.

## 5. Selecting data: `[]`, `.loc`, `.iloc`

Three doors into a DataFrame, each with a distinct contract:

| syntax      | meaning                          | slice rule          |
|-------------|----------------------------------|---------------------|
| `df[col]`   | column(s) by NAME                | -                   |
| `.loc`      | rows/cols by LABEL or boolean    | **inclusive** of end |
| `.iloc`     | rows/cols by POSITION (0-based)  | **exclusive** of end |

### 5.1 Column selection with brackets

In [13]:
age = titanic["age"]            # single name -> Series
subset = titanic[["age", "fare"]]  # LIST of names -> DataFrame (note double [])

print(type(age).__name__, age.name, "| shape:", age.shape)
print(type(subset).__name__, "| shape:", subset.shape)

# Dot access works for simple names...
print(titanic.age.head(3).tolist(), "... convenient but fragile:")

# ⚠️ Common pitfall: dot access breaks for names with spaces, leading digits,
# or Python keywords, and tools can't see the column name as a string.
# Rule of thumb: dot access in quick interactive sessions, BRACKETS everywhere else.

Series age | shape: (891,)
DataFrame | shape: (891, 2)
[22.0, 38.0, 26.0] ... convenient but fragile:


### 5.2 `.loc` - select by LABEL

In [14]:
people = pd.DataFrame(
    {"score": [90, 85, 70]},
    index=["ann", "ben", "cal"],
)
print(people, "\n")

print(people.loc["ben"], "\n")          # one label -> row as Series
print(people.loc[["ann", "cal"]], "\n") # list of labels -> those rows

# Label slices INCLUDE the endpoint (unlike Python lists!):
print(people.loc["ann":"cal"])

     score
ann     90
ben     85
cal     70 

score    85
Name: ben, dtype: int64 

     score
ann     90
cal     70 

     score
ann     90
ben     85
cal     70


In [15]:
# ⚠️ Common pitfall previewed: with INTEGER labels, .loc slices by label, not
# position -- a famous source of off-by-one confusion.
weird = pd.DataFrame({"score": [10, 20, 30]}, index=[10, 20, 30])

print(weird.loc[10:20], "  <- .loc: LABELS 10 and 20, inclusive\n")
print(weird.iloc[0:2],    "  <- .iloc: POSITIONS 0 and 1, exclusive end")

    score
10     10
20     20   <- .loc: LABELS 10 and 20, inclusive

    score
10     10
20     20   <- .iloc: POSITIONS 0 and 1, exclusive end


In [16]:
# .loc takes [row_selector, column_selector] -- rows AND columns in one shot.
first_three = titanic.loc[:2, ["sex", "age", "fare"]]   # labels 0,1,2 (inclusive!)
print(first_three.head())

# A boolean ARRAY in the row slot = filtering (deep dive next section).
adults = titanic.loc[titanic["age"] >= 18, ["sex", "pclass", "fare"]]
print("\nadults shape:", adults.shape)

      sex   age     fare
0    male  22.0   7.2500
1  female  38.0  71.2833
2  female  26.0   7.9250

adults shape: (601, 3)


> 💡 **Pro tip:** need exactly ONE scalar? `.at[row, col]` (labels) and
> `.iat[i, j]` (positions) are faster micro-versions of `.loc`/`.iloc`.

In [17]:
# .iloc = pure integer positions, C-style half-open slices like ordinary lists.
print(titanic.iloc[:3, :2], "\n")   # rows 0-2, cols 0-1 (end EXCLUSIVE)

# Negative positions work like list indexing:
print(titanic.iloc[-2:, -2:])       # last two rows, last two columns

# Single cell:
print("cell [5, 'fare'] by position:", titanic.iloc[5, 7])

   survived  pclass
0         0       3
1         1       1
2         1       3 

    alive  alone
889   yes   True
890    no   True
cell [5, 'fare'] by position: Q


### 5.3 Which selector when? (head-to-head)

| you want...                          | reach for              |
|--------------------------------------|------------------------|
| one column                           | `df["col"]`            |
| several columns                      | `df[["a", "b"]]`       |
| rows by their LABEL range            | `df.loc["a":"c"]` (inclusive) |
| first/last N rows by position        | `df.iloc[:5]`, `df.iloc[-5:]` |
| filter + pick columns in one step    | `df.loc[mask, cols]`   |
| one scalar, hot path                 | `df.at[r, c]` / `df.iat[i, j]` |

> ⚠️ **Common pitfall:** `df[0:5]` on a label-indexed frame slices by
> POSITION (special case!), while `df["0":"5"]` would be labels. When in
> doubt, say `.loc` or `.iloc` explicitly - future-you will thank you.

## 6. Boolean filtering

WHAT: compare a column -> get a True/False mask -> index with it.
WHY:  "give me the rows WHERE..." is the heart of everyday analysis.

In [18]:
# Step 1: build the mask. Step 2: apply it. Keep the steps separate at first --
# it makes debugging much easier.
mask_female = titanic["sex"] == "female"
print(mask_female.head(3), "\n")

females = titanic[mask_female]
print("female passengers:", len(females))

# Combine masks with & (AND), | (OR), ~ (NOT) -- NEVER Python's and/or/not.

# ⚠️ Common pitfall: comparisons bind TIGHTER than & and |, so parentheses are
# MANDATORY around each condition. Without them you get a confusing TypeError.
surviving_women_third = titanic[
    (titanic["sex"] == "female") & (titanic["pclass"] == 3) & (titanic["survived"] == 1)
]
print("surviving women in 3rd class:", len(surviving_women_third))

# ~ flips a mask: everyone who did NOT pay in USD-top-fare bracket below.
not_cheap = titanic[~(titanic["fare"] < 10)]
print("fares >= 10:", len(not_cheap))

0    False
1     True
2     True
Name: sex, dtype: bool

 

female passengers: 314
surviving women in 3rd class: 72
fares >= 10: 555


In [19]:
# Handy predicate helpers:
ports = titanic[titanic["embarked"].isin(["C", "Q"])]        # membership set
print("boarded at C or Q:", len(ports))

mid_fares = titanic[titanic["fare"].between(20, 50)]         # inclusive range
print("20 <= fare <= 50:", len(mid_fares))

# A quick taste of .str methods (full suite in the Advanced notebook):
emails = pd.Series(["ann@x.com", "bob@spam.net", "carol@x.com"])
x_folk = emails[emails.str.startswith("c")]                  # pattern match
print(x_folk.tolist())

boarded at C or Q: 245
20 <= fare <= 50: 216
['carol@x.com']


In [20]:
# query(): write the filter as a STRING -- often the most readable option for
# complex conditions. Column names become variables inside the expression.
result = titanic.query("pclass == 1 and fare > 80 and sex == 'female'")
print(result[["sex", "pclass", "fare"]].head())

# 💡 Pro tip: query shines in method chains (see the Advanced notebook) and
# supports 'in', 'not in', 'between'-style logic via plain English-ish syntax.

        sex  pclass      fare
31   female       1  146.5208
88   female       1  263.0000
195  female       1  146.5208
215  female       1  113.2750
230  female       1   83.4750


## 7. Sorting and ranking

WHAT: order rows by column values (`sort_values`) or by index labels
(`sort_index`); `rank` converts values into competition-style ranks.
WHY:  "top 5 biggest..." questions always start with a sort.

In [21]:
by_fare = titanic.sort_values(by="fare", ascending=False, na_position="last")
print(by_fare[["sex", "pclass", "fare"]].head(3))

# Multiple keys: list the columns, list the directions (parallel lists).
two_keys = titanic.sort_values(
    by=["pclass", "fare"], ascending=[True, False]
)[["pclass", "fare"]]
print("\ncheapest fare within EACH class comes last:\n", two_keys.groupby("pclass").tail(1))

# sort_index puts rows back in index order -- handy after shuffling/sampling:
shuffled = titanic.sample(frac=1, random_state=42)   # frac=1 -> ALL rows, shuffled
restored = shuffled.sort_index()
print("\nsample+sort_index reproduces the original head:")
print(restored.head(3)[["sex", "age"]])

        sex  pclass      fare
679    male       1  512.3292
258  female       1  512.3292
737    male       1  512.3292

cheapest fare within EACH class comes last:
      pclass  fare
822       1   0.0
732       2   0.0
597       3   0.0

sample+sort_index reproduces the original head:
      sex   age
0    male  22.0
1  female  38.0
2  female  26.0


In [22]:
# rank(pct=True): fraction of values <= this one (0..1]. Max fare -> 1.0.
fare_pct = titanic["fare"].rank(pct=True)

top3 = titanic.sort_values("fare", ascending=False).head(3)
top3 = top3.assign(fare_percentile=fare_pct[top3.index])   # aligns by row label!
print(top3[["sex", "pclass", "fare", "fare_percentile"]])

# 💡 rank(method=...) matters with ties: try "average" (default), "min",
# "dense", "first" to control how equal values are handled.

        sex  pclass      fare  fare_percentile
679    male       1  512.3292         0.998878
258  female       1  512.3292         0.998878
737    male       1  512.3292         0.998878


## 8. Derived columns: `[]` assignment and `assign()`

In [23]:
# Direct assignment creates (or OVERWRITES) a column in place.
titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1  # + self
titanic["log_fare"] = np.log1p(titanic["fare"])   # log1p handles fare==0 safely
titanic[["family_size", "log_fare"]].head()

# ⚠️ Common pitfall: direct assignment MUTATES the DataFrame in place. If the
# frame came from a filter chain you may be editing a throwaway copy. Prefer
# .copy() once, then assign -- or better, the functional style below.

,family_size,log_fare
0,2,2.110213
1,2,4.280593
2,1,2.188856
3,2,3.990834
4,1,2.202765


In [24]:
# assign() RETURNS A NEW frame with added/updated columns -- chain-friendly.
# lambdas receive the frame-so-far as 'd', so later columns can use earlier ones.
enriched = (
    titanic
    .assign(
        family_size=lambda d: d["sibsp"] + d["parch"] + 1,
        fare_per_person=lambda d: d["fare"] / d["family_size"],
    )
    .query("fare_per_person > 0")          # drops fare==0 records cleanly
)
enriched[["sex", "pclass", "family_size", "fare_per_person"]].head()

,sex,pclass,family_size,fare_per_person
0,male,3,2,3.62500
1,female,1,2,35.64165
2,female,3,1,7.92500
3,female,1,2,26.55000
4,male,3,1,8.05000


## 9. Dropping, renaming, and index surgery

WHAT: reshape the table's skeleton -- remove rows/columns, rename headers,
promote a column to (or demote from) the index.
WHY:  tidy inputs make every later step simpler.

In [25]:
# drop() defaults to rows (axis=0); axis=1 / columns= targets columns.
# errors="ignore" means 'don't complain if it's already gone' -- safe cleanup.
slim = titanic.drop(columns=["deck"], errors="ignore")   # deck is ~77% missing anyway
row_dropped = slim.drop(index=[0, 1])                    # remove rows by label
axis_style = slim.drop(["deck"], axis=1, errors="ignore")  # older equivalent style
print(slim.shape, "->", row_dropped.shape)

renamed = slim.rename(columns={"embarked": "port_code"})  # mapper dict, returns COPY
print(list(renamed.columns)[:8], "...")

(891, 16) -> (889, 16)
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'port_code'] ...


In [26]:
# set_index promotes a column to row labels; reset_index reverses it.
people = pd.DataFrame(
    {"dept": ["Eng", "Sales", "Eng"], "salary": [95, 70, 88]},
    index=["ann", "ben", "cal"],
)
by_name = people.set_index("dept")        # wait -- set_index on a COLUMN works too;
print(by_name, "\n")                      # duplicate labels are allowed but slower lookups

back_to_columns = by_name.reset_index()   # index becomes a normal column again
print(back_to_columns)

# reset_index(drop=True) THROWS the old index away -- ideal after filtering,
# when leftover non-consecutive labels (3, 17, 214...) get confusing.
sampled = titanic.sample(3, random_state=1)
clean_labels = sampled.reset_index(drop=True)
print(clean_labels.index.tolist())        # boring old 0,1,2 -- exactly what we want

       salary
dept         
Eng        95
Sales      70
Eng        88

 

    dept  salary
0    Eng      95
1  Sales      70
2    Eng      88
[0, 1, 2]


## 10. Foundations checklist (head-to-head recap)

| task                        | idiomatic call                                   |
|-----------------------------|--------------------------------------------------|
| peek at data                | `head / tail / sample / info / describe`         |
| one column / many columns   | `df["col"]` / `df[["a","b"]]`                    |
| rows by label               | `df.loc[label_or_mask, cols]` (inclusive slices) |
| rows by position            | `df.iloc[i:j, k]` (exclusive slices)             |
| complex filter              | `(cond1) & (cond2)` or `df.query(...)`           |
| membership / range          | `isin([...])` / `between(lo, hi)`                |
| top-N questions             | `sort_values(...)` (+ `.head(n)`)                |
| percentile-of-value         | `Series.rank(pct=True)`                          |
| add columns functionally    | `assign(col=lambda d: ...)`                      |
| delete / rename             | `drop(...)`, `rename(columns=mapper)`            |
| index gymnastics            | `set_index` / `reset_index(drop=...)`            |

> 💡 **Pro tip:** press Tab after `df.` in Jupyter to explore attributes, and
> put `?` after any call (e.g. `df.sort_values?`) to read its docs inline.

## Summary & key takeaways

- **pandas = labeled, heterogeneous, messy-data-friendly tables** - Series
  (one labeled column) compose into DataFrames.
- Operations **align on index labels**; mismatched labels yield NaN, not
  errors. Use `.add(other, fill_value=...)` to control that.
- Load with `read_csv(sep=..., header=..., index_col=..., dtype=...,
  na_values=..., parse_dates=...)` - fix messiness at the door.
- Inspect FIRST: `info()`, `describe(include="all")`,
  `value_counts(dropna=False)`, `nunique()`, `memory_usage(deep=True)`.
- Selection contracts: `[]` for columns, `.loc` = labels (**inclusive**
  slices, boolean masks welcome), `.iloc` = positions (**exclusive**).
- Filter with `& | ~` - parentheses mandatory; `isin`, `between`, and
  `query()` cover the everyday special cases.
- Sort with multi-key `sort_values(ascending=[...])`; `rank(pct=True)`
  answers "how big is this value relative to the rest?"
- Prefer `assign()` chains for derived columns; `drop`/`rename`/`reset_index`
  return copies, keeping your pipeline predictable.

**Next up:** `02_pandas_intermediate.nb.py` - missing data, groupby deep dive,
reshaping, and combining tables.